# QLoRA 微調大型語言模型：把 TinyLlama 基礎模型訓練成會聊天的助理

---

## 這份程式要做什麼

把一個**只會接續文字**的基礎模型，微調成**能回答問題的聊天助理**，而且只用一張小 GPU。

```
TinyLlama-1.1B 基礎模型（base）            ← 只會「接下去寫」，不會「回答問題」
        │
        ① 載入時壓縮成 4 位元（量化）           → 省記憶體
        ② 凍結原本的權重，只訓練小小的 LoRA 附加層   → 省運算與記憶體
        ③ 用 3000 筆 UltraChat 對話做監督式微調（SFT）
        │
        ④ 存下 LoRA 附加層，再「合併」回原模型
        ▼
會聊天的 TinyLlama（能回答 "Tell me something about LLMs"）
```

## 先搞懂四個關鍵詞

### 1. 基礎模型 vs 聊天模型

| | 基礎模型（base） | 聊天模型（chat） |
|---|---|---|
| 訓練目標 | 預測下一個詞 | 在對話格式下回答問題 |
| 給它一個問題 | 可能接著寫出更多問題 | 給出回答 |

這份程式用的底座是**基礎模型** `TinyLlama-1.1B-intermediate-step-1431k-3T`，我們要讓它學會「對話」。

### 2. SFT（Supervised Fine-Tuning，監督式微調）

給模型大量「問題 → 好的回答」範例，讓它學會模仿。這裡用 UltraChat 的對話資料。

### 3. LoRA（Low-Rank Adaptation）

全參數微調要更新全部 11 億個參數，很貴。LoRA 的做法是：

```
原本：  輸出 = W · x                       （W 是巨大的權重矩陣，凍結不動）
LoRA：  輸出 = W · x + (α/r) · B · A · x    （只訓練 A 和 B 兩個小矩陣）
```

- `W` 假設是 2048 × 2048，約 420 萬個參數
- `A`（r × 2048）和 `B`（2048 × r）當 r = 64 時，只有約 26 萬個參數
- 訓練時只更新 A 和 B；W 完全不動

### 4. QLoRA = 量化 + LoRA

| 技術 | 做什麼 |
|---|---|
| 量化（4-bit NF4） | 把凍結的 `W` 壓縮成 4 位元存放，大幅省記憶體 |
| LoRA | 只訓練小矩陣，且這些小矩陣用較高精度 |
| 分頁優化器 | 記憶體吃緊時，自動把優化器狀態暫時搬到 CPU，避免爆顯存 |

### 記憶體粗估

| 做法 | 主要記憶體來源 | 粗估 |
|---|---|---|
| 全參數微調（16 位元權重 + Adam） | 權重 2.2 GB + 梯度 2.2 GB + Adam 狀態約 8.8 GB | **13 GB 以上** |
| **QLoRA** | 4-bit 權重約 0.55 GB + 很小的 LoRA 與其優化器狀態 | **遠小於 2 GB（不含計算時的暫存）** |

## 三種微調做法對照

| | 全參數微調 | LoRA | **QLoRA（這份）** |
|---|---|---|---|
| 更新的參數 | 全部 | 只有附加的小矩陣 | 只有附加的小矩陣 |
| 底座模型精度 | 16/32 位元 | 16 位元 | **4 位元** |
| 記憶體需求 | 最高 | 中 | **最低** |

## 硬體提醒

- `bitsandbytes` 的 4 位元量化需要 **NVIDIA GPU（CUDA）**。
- 這份程式用 **bf16**（`bf16=True`）。


## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `AutoModelForCausalLM` | 載入因果語言模型（逐字預測下一個詞，GPT / Llama 這類） |
| `AutoTokenizer` | 載入分詞器 |
| `BitsAndBytesConfig` | 設定 4 位元量化 |
| `LoraConfig` | 設定 LoRA |
| `SFTTrainer`、`SFTConfig`（來自 `trl`） | 監督式微調的訓練器與專用設定 |
| `AutoPeftModelForCausalLM` | 載入「基礎模型 + LoRA 附加層」 |
| `pipeline` | 一行程式做文字生成 |

這份程式裡**沒有用到**的匯入：`TrainingArguments`、`Trainer`、`prepare_model_for_kbit_training`、`get_peft_model`。它們不影響執行，只是多餘的（`SFTConfig` 取代了 `TrainingArguments`；LoRA 由 `SFTTrainer` 自動套用，不需要自己呼叫 `get_peft_model`）。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from datasets import load_dataset
import torch
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from peft import AutoPeftModelForCausalLM
from transformers import pipeline

## 步驟 1：對話模板（chat template）

### 為什麼出現兩個 TinyLlama

| 變數 | 模型 | 用途 |
|---|---|---|
| `model_id`（Chat 版） | `TinyLlama-1.1B-Chat-v1.0` | **只借它的分詞器來做對話格式**，不載入它的權重 |
| `model_name`（base 版） | `TinyLlama-1.1B-intermediate-step-1431k-3T` | **真正要微調的模型** |

基礎模型本身沒有定義對話格式，所以借用 Chat 版分詞器附帶的「對話模板」，把對話整理成統一的文字格式。

### `format_prompt` 做了什麼

UltraChat 的每筆資料有一個 `messages` 欄位，是一串訊息：

```python
[{"role": "user",      "content": "Tell me about cats."},
 {"role": "assistant", "content": "Cats are small carnivorous mammals..."}]
```

```python
prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
```

| 部分 | 說明 |
|---|---|
| `apply_chat_template` | 依模型的模板，把訊息串接成**一個字串**，並加上角色標記 |
| `tokenize=False` | 回傳字串，而不是轉成數字 id |
| `return {"text": prompt}` | 回傳新的 `text` 欄位，之後訓練就用這一欄 |

格式大致像這樣（和你最後推論用的 `prompt` 是同一種格式，具體細節以模板為準）：

```
<|user|>
Tell me about cats.</s>
<|assistant|>
Cats are small carnivorous mammals...</s>
```

`<|user|>`、`<|assistant|>` 標記是誰在說話，`</s>` 是結束符號。**訓練時用什麼格式，推論時就必須用同樣的格式**，模型才認得。

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

template_tokenizer = AutoTokenizer.from_pretrained(model_id)


def format_prompt(example):

    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)
    return {"text": prompt}

## 步驟 2：載入資料集，並只留下 `text` 欄位

```python
load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
    .shuffle(seed=42)
    .select(range(3000))
```

| 步驟 | 說明 |
|---|---|
| `UltraChat 200k` | 大型多輪對話資料集 |
| `split="test_sft"` | 取其中一份切分（用來做 SFT 的那一份） |
| `.shuffle(seed=42)` | 打亂順序，固定種子讓結果可重現 |
| `.select(range(3000))` | **只取 3000 筆**，讓訓練變快 |

### 最關鍵的一行：`remove_columns=dataset.column_names`

```python
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)
```

這一行就是你之前遇到 `KeyError: 'completion'` 的解法。

**問題的來源**：

UltraChat 原本的欄位我印象中是 `prompt`、`prompt_id`、`messages`。新版 `SFTTrainer` 會**自動偵測資料集格式**：只要看到有 `prompt` 這一欄，就以為這是「prompt-completion」格式（提示 + 回答分成兩欄），接著就去讀 `completion` 欄，結果找不到，就報 `KeyError: 'completion'`。

**解法**：`.map(..., remove_columns=...)` 在套用函式之後，把**原本所有欄位都移除**，只留下函式回傳的 `text`。這樣資料集只剩一欄 `text`，`SFTTrainer` 就會當成一般的「語言模型文字」格式處理。

| 時間點 | 欄位 |
|---|---|
| 載入後 | `prompt`、`prompt_id`、`messages` |
| `map` 之後 | **只有 `text`** |

`dataset.column_names` 在 `map` 被呼叫**之前**就會先取值（Python 先算參數），所以傳進去的是原本的欄位清單。

> 驗證方式：在 `map` 前後各印一次 `dataset.column_names`。

### 這是示範規模

3000 筆、1 個 epoch，足以讓模型學會對話的「樣子」，但不要期待它能達到大型聊天模型的水準。

In [ ]:
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k", split="test_sft")
        .shuffle(seed=42)
        .select(range(3000))
)
dataset = dataset.map(format_prompt, remove_columns=dataset.column_names)

## 步驟 3：設定 4 位元量化並載入基礎模型

```python
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
```

| 參數 | 說明 |
|---|---|
| `load_in_4bit=True` | 載入時把權重壓縮成 4 位元，這是 QLoRA 省記憶體的關鍵 |
| `bnb_4bit_quant_type="nf4"` | 使用 **NF4**（Normal Float 4）格式。權重大多呈常態分佈，NF4 專為此設計，比一般 4 位元整數更準 |
| `bnb_4bit_use_double_quant=True` | **雙重量化**：連「量化用的常數」也再壓縮一次，再省一點記憶體 |
| `bnb_4bit_compute_dtype=torch.bfloat16` | 實際**運算時**暫時還原成 bf16（儲存是 4 位元，計算是 16 位元） |

**重點觀念**：4 位元只用在「儲存」，計算時會暫時還原成較高的精度，所以損失不大。

### fp16 與 bf16 的差別

| | fp16 | bf16 |
|---|---|---|
| 位元數 | 16 | 16 |
| 數值範圍 | 較小，容易溢位 | **和 fp32 一樣大，較穩定** |
| 精度（小數位） | 較高 | 較低 |
| GPU 要求 | 多數 GPU | 較新的 NVIDIA GPU |

你把這份程式改成 bf16，要記得**前後一致**：這裡的 `compute_dtype` 用 `torch.bfloat16`，後面訓練設定也用 `bf16=True`、`fp16=False`。

### 載入模型

| 部分 | 說明 |
|---|---|
| `model_name` | 基礎模型（不是 Chat 版） |
| `quantization_config` | 套用上面的量化設定 |
| `device_map="auto"` | 自動決定模型各部分放在哪個裝置（GPU / CPU） |

### 兩行設定

```python
model.config.use_cache = False
model.config.pretraining_tp = 1
```

| 設定 | 說明 |
|---|---|
| `use_cache = False` | 關掉生成用的快取（KV cache）。訓練不需要它，而且和「梯度檢查點」不相容 |
| `pretraining_tp = 1` | Llama 系列特有的設定。我印象中它控制是否模擬預訓練時的張量平行切分，設成 1 使用一般的計算路徑，是 QLoRA 常見做法。我沒有針對你的版本確認 |

載入時終端機會出現 `Loading weights` 進度條，是正常的。

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.config.pretraining_tp = 1

## 步驟 4：載入訓練用的分詞器

```python
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
```

| 設定 | 說明 |
|---|---|
| `trust_remote_code=True` | 允許執行模型倉庫裡的自訂程式碼。Llama 架構其實不需要，**只應該對你信任的來源開啟** |
| `pad_token = eos_token` | 基礎模型的分詞器沒有補齊符號，常見做法是**借用結束符號**當補齊符號，不用新增詞彙表以外的 token |
| `padding_side = "right"` | 補齊補在**右邊**，這是訓練時的標準做法 |

### 靠左還是靠右補齊

| | 靠右補齊（right） | 靠左補齊（left） |
|---|---|---|
| 適用情境 | **訓練** | **批次生成（推論）** |
| 原因 | 序列從頭開始對齊，位置編號自然 | 生成從右邊接續，補在左邊才能讓真正內容緊貼最右側 |

你這份程式把訓練用的分詞器設成靠右。後面推論只丟**單一個 prompt**，沒有補齊的問題，所以不受影響；如果之後要批次生成多個 prompt，才需要改成靠左。

### 這個分詞器要傳給誰

後面的 `SFTTrainer` 會用 `processing_class=tokenizer` 接收它，所以這裡的 `pad_token`、`padding_side` 設定**會真的被訓練使用**。如果沒有傳 `processing_class`，我印象中訓練器會自己從模型載入一個預設的，你在這裡做的設定就派不上用場。（這點我沒有針對你的版本確認。）

> 進階提醒：`pad_token` 和 `eos_token` 是同一個符號時，有些實作會用「符號是否等於補齊符號」來遮蔽 loss，連帶把真正的結束符號也遮掉，使模型較難學會「何時停止」。新版 TRL 的實際做法我沒有確認，如果你發現模型生成時停不下來，這是值得檢查的方向。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name,  
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## 步驟 5：設定 LoRA

| 參數 | 值 | 說明 |
|---|---|---|
| `r` | 64 | LoRA 的**秩**，也就是小矩陣 A、B 的寬度。越大，能學的東西越多，參數也越多 |
| `lora_alpha` | 32 | 縮放係數。實際縮放是 `lora_alpha / r = 32 / 64 = 0.5` |
| `target_modules` | 7 個名稱 | 要在哪些層加上 LoRA |
| `lora_dropout` | 0.05 | LoRA 路徑的隨機丟棄，輕微正則化 |
| `bias` | `"none"` | 不訓練偏差項 |
| `task_type` | `"CAUSAL_LM"` | 任務類型：因果語言模型 |

### `target_modules`：加在哪些層

```
q_proj, k_proj, v_proj, o_proj      ← 注意力機制的四個線性層
gate_proj, up_proj, down_proj       ← 前饋網路（MLP）的三個線性層
```

這幾乎涵蓋了 Transformer 裡**所有的線性層**。只加在 `q_proj`、`v_proj` 是比較省的做法；這份設定是「全部都加」，效果通常較好，但參數也較多。

### 可訓練參數量粗估

我依 TinyLlama 的架構數字（隱藏維度 2048、前饋維度 5632、22 層、4 個 KV 頭，這些數字是我的印象，沒有逐一確認）和 `r = 64` 算：

```
每個線性層的 LoRA 參數 = r × (輸入維度 + 輸出維度)
每層合計 ≈ 229 萬
22 層合計 ≈ 5046 萬  ≈ 1.1B 的 4.6%
```

也就是只訓練約 **5%** 的參數。實際數字可以用 `trainer.model.print_trainable_parameters()` 確認。

In [ ]:
peft_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=["k_proj", "gate_proj", "up_proj", "q_proj", "v_proj", "o_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 步驟 6：訓練設定（SFTConfig）

`SFTConfig` 是 `TrainingArguments` 的擴充版，除了一般訓練參數，還包含 SFT 專用的設定。

### 一般訓練參數

| 參數 | 值 | 說明 |
|---|---|---|
| `per_device_train_batch_size` | 2 | 每次只放 2 筆進 GPU（顯存有限） |
| `gradient_accumulation_steps` | 4 | 累積 4 次梯度才更新一次，**等效 batch size = 2 × 4 = 8** |
| `optim` | `"paged_adamw_32bit"` | 分頁版 AdamW。記憶體吃緊時，把優化器狀態暫時換到 CPU，避免爆顯存 |
| `learning_rate` | `2e-4` | 比全參數微調（約 `2e-5`）**大約 10 倍**：LoRA 只訓練新加的小矩陣，可以承受較大的學習率 |
| `bf16` / `fp16` | True / False | 使用 bf16 混合精度（要和量化設定的 `compute_dtype` 一致） |
| `lr_scheduler_type` | `"cosine"` | 學習率依餘弦曲線逐漸下降 |
| `num_train_epochs` | 1 | 整份資料看 1 遍 |
| `logging_steps` | 10 | 每 10 步印一次 loss |
| `gradient_checkpointing` | True | 不保存全部中間結果，需要時重算，用**時間換記憶體** |

步數速算：3000 筆 ÷ 等效 batch 8 = **375 次參數更新**，每 10 步記錄一次，共約 37 筆 loss 記錄。

### SFT 專用參數（從 `SFTTrainer` 搬到這裡）

| 參數 | 值 | 說明 |
|---|---|---|
| `max_length` | 512 | 每筆資料最多 512 個 token，超過的截斷（舊版叫 `max_seq_length`）。**顯存小時這個值很重要**，因為記憶體用量和序列長度成正比。我印象中不設的話預設值較大（約 1024），可能讓 8 GB 的 GPU 吃緊 |
| `dataset_text_field` | `"text"` | 訓練時讀取 `text` 這一欄（步驟 2 建立的） |

### 梯度累積是什麼

顯存放不下大 batch，就「分次算、累積起來、一次更新」：

```
batch 2 → 算梯度，先不更新
batch 2 → 算梯度，加進去
batch 2 → 算梯度，加進去
batch 2 → 算梯度，加進去 → 現在才更新一次（等效 batch = 8）
```

效果接近真的用 batch 8，但記憶體只需要 batch 2。

In [ ]:
output_dir = "./qlora_results"
training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    gradient_checkpointing=True,
    max_length=512,
    dataset_text_field="text",
)

## 步驟 7：建立 SFTTrainer

```python
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
)
```

| 參數 | 說明 |
|---|---|
| `model` | 已量化的基礎模型 |
| `train_dataset` | 整理好、只剩 `text` 欄位的 3000 筆資料 |
| `args` | 剛剛的 `SFTConfig` |
| `peft_config` | 傳入 LoRA 設定。`SFTTrainer` 會**自動把模型包上 LoRA 附加層**，所以不需要自己呼叫 `get_peft_model` |
| `processing_class` | 分詞器（新版名稱；舊版叫 `tokenizer`） |

### 建立 Trainer 時會發生的事

終端機會出現類似這樣的訊息，都是**正常的前處理**：

```
Adding EOS to train dataset      ← 幫每筆文字補上結束符號
Tokenizing train dataset         ← 用分詞器把文字轉成 token（並依 max_length 截斷）
```

> 你之前的錯誤就是發生在 `Tokenizing train dataset` 這一步，原因見步驟 2。

### 值得留意：結束符號可能重複

對話模板產生的文字，結尾本來就有 `</s>`，而 `SFTTrainer` 又會「補上結束符號」。我印象中它會先檢查結尾是否已經有，但這份資料的結尾可能是 `</s>` 加換行，檢查不一定會視為已有。實際情況沒有確認，可以印出幾筆 token 看看。

### SFT 預設的學習方式

一般情況下，`SFTTrainer` 對整段文字（使用者的提問加上助理的回答）都計算 loss，而不是只學助理回答的部分。新版 TRL 有提供只對回答部分計算 loss 的選項，名稱依版本而定，我沒有確認。

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
)   

## 步驟 8：訓練並儲存 LoRA 附加層

```python
trainer.train()
trainer.save_model("LoRA_Fineune_model")
```

### `trainer.train()`

- **凍結**：4 位元的基礎模型權重不動
- **更新**：只有 LoRA 的 A、B 矩陣
- 約 375 次更新，終端機每 10 步會印出 loss，**loss 逐漸下降**代表模型在學習

### 存的是什麼

存的**只有 LoRA 附加層**（外加一份設定檔，記錄它是掛在哪個基礎模型上），不是整個模型：

| 項目 | 大小 |
|---|---|
| 基礎模型（1.1B） | 數 GB |
| LoRA 附加層 | 我粗估約 100~200 MB（約 5000 萬個參數），沒有實測 |

同一個基礎模型可以搭配多組小小的附加層，分別用於不同任務，這是 LoRA 的一大好處。

### 為什麼用 `trainer.save_model`

`save_model` 是 **`Trainer` 物件**的方法，不是模型的方法。舊寫法 `trainer.model.save_model(...)` 是對模型呼叫，模型沒有這個方法。你改成 `trainer.save_model(...)` 是正確的。因為訓練器管理的是被 LoRA 包裝過的模型，存下來的會是附加層。`Trainer` 存檔時通常也會一併存下分詞器，這點我沒有針對你的版本確認。

In [ ]:
trainer.train()
trainer.save_model("LoRA_Fineune_model")

## 步驟 9：載入並合併 LoRA

```python
model = AutoPeftModelForCausalLM.from_pretrained(
    "LoRA_Fineune_model",
    device_map="auto",
    low_cpu_mem_usage=True,
)
merged_model = model.merge_and_unload()
```

### 載入

`AutoPeftModelForCausalLM` 讀取附加層資料夾裡的設定檔，知道它掛在哪個基礎模型上，會**自動把基礎模型也載入**，再把附加層接上去。所以基礎模型要能被找到（本機快取或網路）。

| 參數 | 說明 |
|---|---|
| `device_map="auto"` | 自動放到可用的裝置 |
| `low_cpu_mem_usage=True` | 載入時節省 CPU 記憶體 |

### `merge_and_unload()`：合併

```
合併前：  輸出 = W · x + (α/r) · B · A · x       ← 要同時算兩條路
合併後：  W' = W + (α/r) · B · A                  ← 把附加層直接加進權重
         輸出 = W' · x                             ← 回到一條路
```

| | 合併前 | 合併後 |
|---|---|---|
| 結構 | 基礎模型 + LoRA 附加層 | 一個普通的 transformers 模型 |
| 推論 | 多一條路徑，略慢 | 和一般模型一樣快 |
| 依賴 | 需要 `peft` | 不需要 `peft` |
| 可以還原嗎 | 附加層獨立，隨時可換 | 合併後就**混在一起**，不能分離 |

> 這裡載入時沒有指定量化設定，所以基礎模型是以一般精度載入（不是 4 位元），合併才能正常進行。精度和記憶體用量依你的版本預設而定，我沒有確認。

### 顯存提醒（小 GPU）

這一格是在**訓練用的模型還留在記憶體**的情況下，再載入一份完整的模型。`trainer` 仍然持有訓練時的模型，所以 8 GB 這種小顯存可能吃緊。如果遇到 `CUDA out of memory`，可以在這一格之前先釋放：

```python
import gc
del trainer, model
gc.collect()
torch.cuda.empty_cache()
```

在 terminal 用 `python test.py` 執行則不用擔心跨次執行的殘留，只要考慮同一次執行裡的峰值。

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "LoRA_Fineune_model",
    device_map="auto",
    low_cpu_mem_usage=True,
)

merged_model = model.merge_and_unload()

## 步驟 10：用微調後的模型生成回答

```python
prompt = """
<|user|>
Tell me something about Large Language Models (LLMs).
</s>
<|assistant|>
"""
```

這個 `prompt` 就是**訓練時用的對話格式**：使用者那一輪結束後，接上 `<|assistant|>`，讓模型接著寫助理的回答。**格式一定要和訓練時一致**，模型才知道現在輪到它回答。

```python
pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])
```

| 部分 | 說明 |
|---|---|
| `pipeline("text-generation")` | 現成的文字生成工具 |
| `[0]["generated_text"]` | 回傳的是 list，取第一個結果的文字 |
| 輸出內容 | **包含你的 prompt 加上模型生成的續寫** |

### 輸出長度與隨機性

這份程式沒有指定生成參數，所以：

- **長度**：用預設的生成長度上限。如果回答被截得很短，加上 `max_new_tokens=256`。預設值依模型的設定而定，我沒有確認
- **隨機性**：預設通常是**貪婪解碼**（每次選機率最高的詞），同樣的輸入每次輸出相同。想要多樣化，加 `do_sample=True` 和 `temperature=0.7`

```python
pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
```

### 怎麼判斷微調有沒有效

最好的方法是**對照**：用同一個 prompt 測試**微調前的基礎模型**，比較兩者的輸出。基礎模型常常不會好好回答，可能接著寫更多問題或不相關的內容；微調後則應該像在回答問題。這個預期是我依方法特性推測的，請以你跑出來的結果為準。

In [ ]:
prompt = """
<|user|>
Tell me something about Large Language Models (LLMs).
</s>
<|assistant|>
"""

pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
)

print(pipe(prompt)[0]["generated_text"])

---

# 重點整理

## 一、整份程式一句話

> 把基礎模型 TinyLlama 以 4 位元量化載入，在所有線性層加上 LoRA 小矩陣，用 3000 筆只剩 `text` 欄位的 UltraChat 對話做一個 epoch 的監督式微調，最後把 LoRA 合併回模型，用對話格式測試生成。

## 二、必須搞懂的 11 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 基礎模型 vs 聊天模型 | 前者只會接續文字，後者學過對話格式 |
| 2 | SFT | 用「問題 → 好回答」的範例教模型模仿 |
| 3 | 對話模板 | `apply_chat_template` 把對話整理成帶角色標記的字串 |
| 4 | LoRA | 凍結原權重，只訓練兩個小矩陣 A、B |
| 5 | `r` 與 `lora_alpha` | `r` 決定 LoRA 容量；縮放 = `lora_alpha / r` |
| 6 | 量化（NF4） | 把凍結的權重壓成 4 位元儲存，計算時還原 |
| 7 | QLoRA | 量化 + LoRA，能在小 GPU 微調大模型 |
| 8 | 梯度累積 | 小 batch 分次算、累積後才更新，模擬大 batch |
| 9 | 梯度檢查點 | 用重算換記憶體 |
| 10 | `merge_and_unload` | 把 LoRA 加回權重，變成普通模型 |
| 11 | 資料集格式偵測 | `SFTTrainer` 看欄位名稱決定資料格式（有 `prompt` 欄會被當成 prompt-completion） |

## 三、程式流程速記

```
1. template_tokenizer（Chat 版）→ 只借它的對話模板
2. load_dataset + map(remove_columns) → 3000 筆對話，只留 text 欄位
3. BitsAndBytesConfig（bf16）      → 4 位元 NF4 量化設定
4. from_pretrained（base 版）      → 載入量化後的基礎模型
5. tokenizer 設定                  → pad = eos、靠右補齊
6. LoraConfig                      → r=64、7 種線性層、alpha=32
7. SFTConfig                       → batch 2 × 累積 4、lr 2e-4、bf16、max_length=512
8. SFTTrainer + train + save_model → 只訓練 LoRA，存附加層
9. AutoPeftModel + merge           → 載入並合併成完整模型
10. pipeline 生成                  → 用對話格式測試
```

## 四、最容易搞混的地方

1. **為什麼出現兩個 TinyLlama？**
   Chat 版只借它的分詞器取得對話模板；真正訓練的是 base 版。

2. **4 位元的模型怎麼訓練？**
   不是訓練 4 位元的權重。權重凍結不動，訓練的是額外加上的 LoRA 小矩陣（精度較高）。

3. **為什麼 `map` 要加 `remove_columns`？**
   原資料有 `prompt` 欄，會讓 `SFTTrainer` 誤判成 prompt-completion 格式，去找不存在的 `completion` 欄而報錯。

4. **`SFTConfig` 和 `TrainingArguments` 的關係？**
   `SFTConfig` 繼承 `TrainingArguments`，多了 SFT 專用的參數（`max_length`、`dataset_text_field` 等）。

5. **bf16 要改哪些地方？**
   量化設定的 `compute_dtype` 和訓練設定的 `bf16` / `fp16` 要一致。

6. **`r` 越大越好嗎？**
   不一定。`r` 大，容量大、參數多、更吃資源；小資料時太大反而容易過擬合。

7. **為什麼學習率比全參數微調大？**
   只訓練新加的小矩陣，不會破壞原權重，可承受較大學習率。

8. **存下來的 LoRA 能單獨使用嗎？**
   不能，必須搭配原本的基礎模型。

9. **合併後還能拆開嗎？**
   不能，合併是不可逆的。需要保留彈性就只存 LoRA。

10. **推論的 prompt 為什麼格式很重要？**
    模型是在特定格式下學會對話的，格式不一致就認不得。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 處理 |
|---|---|---|
| `TypeError: ... unexpected keyword argument 'tokenizer'` | 新版 TRL 把 `tokenizer` 改名為 `processing_class` | 改用 `processing_class=tokenizer` |
| `TypeError: ... unexpected keyword argument 'max_seq_length'` | 這個參數搬到 `SFTConfig`，且改名為 `max_length` | 在 `SFTConfig` 裡設 `max_length` |
| `KeyError: 'completion'` | 資料集仍保留 `prompt` 欄位，被誤判成 prompt-completion 格式 | `map(..., remove_columns=dataset.column_names)`，只留 `text` |
| `CUDA out of memory` | 顯存不足 | 降低 `max_length`、保持小 batch、確認有開量化與 `gradient_checkpointing`、載入合併前先釋放訓練用的模型 |
| bf16 相關錯誤 | GPU 不支援 bf16 | 改回 `fp16=True`，並把 `compute_dtype` 改回 `torch.float16` |
| bitsandbytes 相關錯誤 | 沒有 CUDA GPU 或版本不相容 | 確認有 NVIDIA GPU 與對應的驅動、套件版本 |
| 載入合併時找不到基礎模型 | 附加層設定指向的基礎模型不在快取，也無法連網 | 連網下載一次，或確認路徑 |
| 生成的回答很短 | 預設生成長度上限太小 | 加 `max_new_tokens` |
| 生成停不下來 | 結束符號沒學好 | 檢查 `pad_token = eos_token` 的影響與資料末端的 EOS |
| 生成的內容亂跑或不像在回答 | prompt 格式和訓練不一致 | 確認用的是同一種對話格式 |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **多餘的匯入**
   `TrainingArguments`、`Trainer`、`prepare_model_for_kbit_training`、`get_peft_model` 都沒用到。

2. **`pad_token = eos_token` 的副作用**
   有些實作會連真正的結束符號一起遮蔽，影響「何時停止」的學習。

3. **結束符號可能重複**
   模板結尾已有 `</s>`，`SFTTrainer` 又會補一個。

4. **預設對整段文字計算 loss**
   使用者的提問也被當成學習目標，不只是助理的回答。

5. **資料量很小**
   只有 3000 筆、1 個 epoch，主要是示範流程。

6. **沒有驗證資料，也沒有訓練中評估**
   只能靠 loss 曲線和最後的生成結果主觀判斷。

7. **`trust_remote_code=True` 不一定需要**
   Llama 架構是內建支援的，這個選項會執行來源的自訂程式碼，只應對可信任的來源開啟。

8. **載入合併時沒指定精度**
   實際精度和記憶體用量依版本預設而定。

9. **資料夾名稱拼字 `Fineune`**
   前後拼法一致，不影響執行。

10. **在 notebook 中重複執行**
    LoRA 會直接加進 `model` 物件裡，重新建立 `SFTTrainer` 前，最好從「載入模型」那一格重新執行，或重啟 Kernel，避免重複套用 LoRA（這是我依 peft 的運作方式推論的）。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 檢查資料集欄位（驗證 remove_columns 的效果）
print(dataset.column_names)          # 預期只有 ['text']
print(dataset[0]["text"][:300])      # 看對話模板長什麼樣
print(dataset[0]["text"][-30:])      # 看結尾有沒有 </s>

# 2. 確認可訓練參數量
trainer.model.print_trainable_parameters()

# 3. 對照：用同一個 prompt 測試「微調前的基礎模型」
base = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
pipe_base = pipeline("text-generation", model=base, tokenizer=tokenizer)
print(pipe_base(prompt, max_new_tokens=100)[0]["generated_text"])

# 4. 調整 LoRA 的 r（8、16、64），比較訓練時間與生成品質

# 5. 只在 q_proj、v_proj 加 LoRA（較省），看效果差多少

# 6. 把 3000 筆增加到 10000 筆，看生成品質有沒有明顯提升

# 7. 把合併後的完整模型存下來，之後不依賴 peft 也能直接使用
merged_model.save_pretrained("merged_tinyllama")
tokenizer.save_pretrained("merged_tinyllama")
```

## 八、與前面學過的微調方式對照

| | 影評分類 | NER | MLM 繼續預訓練 | SetFit | **QLoRA（這份）** |
|---|---|---|---|---|---|
| 模型類別 | `...ForSequenceClassification` | `...ForTokenClassification` | `...ForMaskedLM` | Sentence Transformer + 分類器 | **`...ForCausalLM`** |
| 模型規模 | 約 1 億 | 約 1 億 | 約 1 億 | 約 1 億 | **約 11 億** |
| 更新的參數 | 全部 | 全部 | 全部 | 全部 | **只有 LoRA（約 5%）** |
| 精度 | 16/32 位元 | 16/32 位元 | 16/32 位元 | 16/32 位元 | **4 位元量化** |
| 任務 | 判斷類別 | 標註實體 | 填空 | 少樣本分類 | **生成回答** |
| 關鍵技術 | 分類頭 | 標籤對齊 | 遮蔽 | 配對放大 | **量化 + LoRA** |

共通的主軸不變：**從預訓練權重出發，用任務資料與損失函數，呼叫 `train()` 更新權重**。差別只在「更新哪些參數」和「用什麼精度存放」。

## 九、一句話記住

> **QLoRA = 把大模型壓成 4 位元凍結起來，只訓練外掛的小矩陣。** 它讓你在小 GPU 上，用很少的資源，把基礎模型微調成會聊天的助理。